# Assignment 2

## Bayesian Classifier

Complete the following without AI assistance. You can Google and lookup functions for pandas and numpy that make the calculations easier. Also, you can ask me :)

In [29]:
import numpy as np

def compute_priors(y):
    priors = y.value_counts(normalize=True)
    #print(priors)
    #this line below just formats the dictionary nicely
    priors.index = [f"{y.name}={i}" for i in priors.index]
    priors = priors.to_dict()
    return priors

def specific_class_conditional(x,xv,y,yv):
    prob = ((x == xv) & (y == yv)).sum() / (y==yv).sum()
    return prob

def class_conditional(X,y):
    probs = {}
    for cols in X.columns:
        for val in X[cols].unique():
            for val2 in y.unique():
                probs[f"{cols}={val}|{y.name}={val2}"] = specific_class_conditional(X[cols],val,y,val2)
    return probs

def posteriors(probs,priors,x):
    post_probs = {}
    total_sum =0
    #print(x.index)
    for i in priors.keys():
        #print(priors[i])
        post_probs[i] = priors[i]
        for j in x.index:
           # print("Post Probs")
            #print(post_probs[i])
            #print(f"{j}={x[j]}")
            try:
                post_probs[i] *= probs[f"{j}={x[j]}|{i}"]
            except KeyError:
              for i in priors.keys():
                  post_probs[i] = 0.5

    print(post_probs)
    for i in post_probs.values():
        total_sum += i
    for i in post_probs.keys():
        #print(post_probs[i])
        post_probs[i] /= total_sum
    return post_probs

def train_test_split(X,y,test_frac=0.5):
    inxs = list(range(len(y)))
    np.random.shuffle(inxs)
    X = X.iloc[inxs,:]
    y = y.iloc[inxs]
    X_size = int(len(X) * test_frac)
    y_size = int(len(y) * test_frac)
    Xtrain = X.iloc[X_size:]
    ytrain = y.iloc[y_size:]
    Xtest = X.iloc[:X_size]
    ytest = y.iloc[:y_size]

    return Xtrain,ytrain,Xtest,ytest

def exercise_6(Xtrain,ytrain,Xtest,ytest):
    probs_train = class_conditional(Xtrain,ytrain)
    priors_train = compute_priors(ytrain)

    correct = 0
    ypred = []

    for i in Xtest.index:
        posterier_train = posteriors(probs_train,priors_train,Xtest.loc[i])

        max_val = max(posterier_train.values())
        best = [cls for cls, v in posterier_train.items() if v == max_val]

        pred = 0 if len(best) > 1 else best[0]

        if isinstance(pred, str) and "=" in pred:
            pred = int(pred.split("=")[1])

        ypred.append(pred)

        if pred == ytest.loc[i]:
            correct += 1
    accuracy = correct / len(Xtest)
    return accuracy



For this lab, we are going to first implement an empirical naive bayesian classifier, then implement feature importance measures and apply it to a dataset, and finally, we will examine the affect of modifying the priors.

For developing this lab, we can use famous Titanic Kaggle dataset. Description of the data is found https://www.kaggle.com/c/titanic/data.

In [15]:
import pandas as pd
titanic_df = pd.read_csv(
    f"https://raw.githubusercontent.com/Anderson-Lab/csc-466-student/refs/heads/main/data/titanic.csv"
)
titanic_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


We only need a few columns, and I will also perform some preprocessing for you:

In [16]:
features = ['Pclass','Survived','Sex','Age']
titanic_df = titanic_df.loc[:,features]
print('Before')
display(titanic_df.head())
titanic_df.loc[:,'Pclass']=titanic_df['Pclass'].fillna(titanic_df['Pclass'].mode()).astype(int)
titanic_df.loc[:,'Age']=titanic_df['Age'].fillna(titanic_df['Age'].median())
titanic_df.loc[:,'Age']=(titanic_df['Age']/10).astype(str).str[0].astype(int)*10
titranic_df = titanic_df.dropna()
print('After')
titanic_df.head()

Before


,Pclass,Survived,Sex,Age
0,3,0,male,22.0
1,1,1,female,38.0
2,3,1,female,26.0
3,1,1,female,35.0
4,3,0,male,35.0


After


,Pclass,Survived,Sex,Age
0,3,0,male,20.0
1,1,1,female,30.0
2,3,1,female,20.0
3,1,1,female,30.0
4,3,0,male,30.0


In [17]:
titanic_df.describe()

,Pclass,Survived,Age
count,891.000000,891.000000,891.000000
mean,2.308642,0.383838,24.208754
std,0.836071,0.486592,13.562886
min,1.000000,0.000000,0.000000
25%,2.000000,0.000000,20.000000
50%,3.000000,0.000000,20.000000
75%,3.000000,1.000000,30.000000
max,3.000000,1.000000,80.000000


#### Problem 1.
Describe what normalization and data cleaning and engineering was done to the original dataset.

**The data was normalized by having the ages rounded to the nearest whole number. This basically puts the ages into bins of 10**

#### Exercise 1
Create a function to determine the prior probability of ALL the classes in ``y``. The result must be in the form of a Python dictionary such as ``priors = {'Survived=0': 0.4, 'Survived=1': 0.6}``.

In [19]:
survived_priors = compute_priors(titanic_df['Survived'])
survived_priors

{'Survived=0': 0.6161616161616161, 'Survived=1': 0.3838383838383838}

In [20]:
compute_priors(titanic_df['Age'])

{'Age=20.0': 0.44556677890011226,
 'Age=30.0': 0.18742985409652077,
 'Age=10.0': 0.11447811447811448,
 'Age=40.0': 0.09988776655443322,
 'Age=0.0': 0.06958473625140292,
 'Age=50.0': 0.05387205387205387,
 'Age=60.0': 0.02132435465768799,
 'Age=70.0': 0.006734006734006734,
 'Age=80.0': 0.001122334455667789}

In [ ]:
y_example = titanic_df['Age']
y_example.name

'Age'

#### Exercise 2
Create a function to calculate the specific class conditional probability. Assume x and y are pd.Series objects. Assume xv and yv are specific values. This function should return $\Pr(x==xv|y==yv)$.

In [ ]:
prob = specific_class_conditional(titanic_df['Sex'],'female',titanic_df['Survived'],0)
prob

np.float64(0.14754098360655737)

#### Exercise 3
Now construct a dictionary based data structure that stores all possible class conditional probabilities (e.g., loop through all possible combinations of values). The keys in your dictionary should be of the form "pclass=1|survived=0". ``X`` is a ``pd.DataFrame`` object and ``y`` is a ``pd.Series`` object. You can retrieve the name of the series object ``y`` by accessing ``y.name``.

Aside: I know it might be a bit annoying to store the key of this dictionary as a string instead of as say a tuple of tuples, but I think the way this prints for folks learning this is reason enough to use strings in this instance.

In [10]:
X = titanic_df.drop("Survived",axis=1)
X.columns

Index(['Pclass', 'Sex', 'Age'], dtype='object')

In [11]:
X = titanic_df.drop("Survived",axis=1)
y = titanic_df["Survived"]
probs = class_conditional(X,y)
probs

{'Pclass=3|Survived=0': np.float64(0.6775956284153005),
 'Pclass=3|Survived=1': np.float64(0.347953216374269),
 'Pclass=1|Survived=0': np.float64(0.14571948998178508),
 'Pclass=1|Survived=1': np.float64(0.39766081871345027),
 'Pclass=2|Survived=0': np.float64(0.1766848816029144),
 'Pclass=2|Survived=1': np.float64(0.2543859649122807),
 'Sex=male|Survived=0': np.float64(0.8524590163934426),
 'Sex=male|Survived=1': np.float64(0.31871345029239767),
 'Sex=female|Survived=0': np.float64(0.14754098360655737),
 'Sex=female|Survived=1': np.float64(0.6812865497076024),
 'Age=20.0|Survived=0': np.float64(0.48816029143898),
 'Age=20.0|Survived=1': np.float64(0.37719298245614036),
 'Age=30.0|Survived=0': np.float64(0.17122040072859745),
 'Age=30.0|Survived=1': np.float64(0.2134502923976608),
 'Age=50.0|Survived=0': np.float64(0.051001821493624776),
 'Age=50.0|Survived=1': np.float64(0.05847953216374269),
 'Age=0.0|Survived=0': np.float64(0.04371584699453552),
 'Age=0.0|Survived=1': np.float64(0.11

#### Exercise 4
Now you are ready to calculate the posterior probabilities for a given sample. Write and test the following function that returns a dictionary where the keys are of the form "Survived=0|Pclass=1,Sex=male,Age=60". Make sure you return 0.5 if the specific combination of values does not exist. ``probs`` and ``priors`` are defined the same as above. ``x`` is a pd.Series object that represents a specific example/sample from our dataset.

In [21]:
probs = class_conditional(X,y)
print(probs)
priors = compute_priors(y)
x = titanic_df.drop("Survived",axis=1).loc[2]
x

{'Pclass=3|Survived=0': np.float64(0.6775956284153005), 'Pclass=3|Survived=1': np.float64(0.347953216374269), 'Pclass=1|Survived=0': np.float64(0.14571948998178508), 'Pclass=1|Survived=1': np.float64(0.39766081871345027), 'Pclass=2|Survived=0': np.float64(0.1766848816029144), 'Pclass=2|Survived=1': np.float64(0.2543859649122807), 'Sex=male|Survived=0': np.float64(0.8524590163934426), 'Sex=male|Survived=1': np.float64(0.31871345029239767), 'Sex=female|Survived=0': np.float64(0.14754098360655737), 'Sex=female|Survived=1': np.float64(0.6812865497076024), 'Age=20.0|Survived=0': np.float64(0.48816029143898), 'Age=20.0|Survived=1': np.float64(0.37719298245614036), 'Age=30.0|Survived=0': np.float64(0.17122040072859745), 'Age=30.0|Survived=1': np.float64(0.2134502923976608), 'Age=50.0|Survived=0': np.float64(0.051001821493624776), 'Age=50.0|Survived=1': np.float64(0.05847953216374269), 'Age=0.0|Survived=0': np.float64(0.04371584699453552), 'Age=0.0|Survived=1': np.float64(0.1111111111111111), 

,2
Pclass,3
Sex,female
Age,20.0


In [22]:
post_probs = posteriors(probs,priors,x)
post_probs

{'Survived=0': np.float64(0.46699312907215196),
 'Survived=1': np.float64(0.533006870927848)}

Here is one more test you should check. Your code should return 50/50 if given a value that is not in the empirical distribution.

In [ ]:
x = titanic_df.drop("Survived",axis=1).loc[2]
x = x.copy()
x['Age'] = 200
x

,2
Pclass,3
Sex,female
Age,200


In [30]:
post_probs = posteriors(probs,priors,x)
post_probs

{'Survived=0': np.float64(0.030070479949544718), 'Survived=1': np.float64(0.03432121679616636)}
0.030070479949544718
0.03432121679616636


{'Survived=0': np.float64(0.46699312907215196),
 'Survived=1': np.float64(0.533006870927848)}

#### Exercise 5
All this is great, but how would you evaluate how we are doing? Create a function called train_test_split that splits our dataframe into a training and testing dataset. To make sure this is done randomly, I have inserted a shuffle into the code for you. The ``test_frac`` is the fraction of the dataset that will be held out for testing.

In [ ]:
import numpy as np
np.random.seed(2)
Xtrain,ytrain,Xtest,ytest=train_test_split(X,y)
print('Xtrain')
display(Xtrain.head())
print('ytrain')
display(ytrain.head())
print('Xtest')
display(Xtest.head())
print('ytest')
display(ytest.head())

Xtrain


,Pclass,Sex,Age
589,3,male,20.0
716,1,female,30.0
154,3,male,20.0
843,3,male,30.0
24,3,female,0.0


ytrain


,Survived
589,0
716,1
154,0
843,0
24,0


Xtest


,Pclass,Sex,Age
707,1,male,40.0
37,3,male,20.0
615,2,female,20.0
169,3,male,20.0
68,3,female,10.0


ytest


,Survived
707,1
37,0
615,1
169,0
68,1


#### Exercise 6
For this exercise, create a training dataset of size 50% and then using your solutions to previous exercises, find the prediction accuracy for the test dataset. **NOTE:** When/if there is a tie in the posterior probabilities, your code should predict that the passenger passed away.

In [ ]:
ytest

,Survived
495,0
648,0
278,0
31,1
255,1
...,...
603,0
327,1
827,1
102,0


In [ ]:
np.random.seed(0)
Xtrain,ytrain,Xtest,ytest=train_test_split(X,y)
accuracy = exercise_6(Xtrain,ytrain,Xtest,ytest)
accuracy

339
445


0.7617977528089888

That's not bad!

#### Problem 2:
Is that better than guessing all passengers died? What is the accuracy on the test set if you guessed all passengers died?

**This is way better than guessing all passengers died. The accuracy on the test set if I guessed all passengers died is the fraction of passengers in the test set that actaully died.**